# 考试对比：老师 vs n系列最强 vs 蒸馏学生

考试集 = 从已标注图片随机抽 200 张（seed=20261030）。逐个模型跑 mAP，输出排名。


In [ ]:
!pip install ultralytics -q


## 一、定位考试集并生成数据配置（zip 会被 Kaggle 自动解压，找不到 zip 就直接用已解压目录）


In [ ]:
import glob, os, zipfile, yaml

# 诊断：打印 /kaggle/input 挂载的目录与文件数
print("== /kaggle/input 挂载内容 ==")
for root, dirs, files in os.walk("/kaggle/input"):
    if files:
        print(" ", root, "->", len(files), "files")
print("==")

# 模型路径按 slug 命名（数据集上传时文件名都可能是 best.pt，用目录名区分）
def slug_of(p):
    parts = p.replace("\\", "/").split("/")
    for i, x in enumerate(parts):
        if x == "datasets" and i + 2 < len(parts):
            return parts[i + 2]
    return os.path.basename(p)

# 考试集：1) 优先 zip（含 exam）；2) 没有 zip -> Kaggle 已自动解压，找含 data.yaml 的目录
cands_zip = sorted(glob.glob("/kaggle/input/**/*.zip", recursive=True))
print("挂载的 zip:", cands_zip)

data_dir = None
if cands_zip:
    exam_zips = [p for p in cands_zip if "exam" in os.path.basename(p).lower()]
    ZIP = (exam_zips or cands_zip)[0]
    print("考试集 zip:", ZIP)
    EXTRACT = "/kaggle/working/exam"
    if os.path.exists(EXTRACT):
        os.system(f"rm -rf {EXTRACT}")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(EXTRACT)
    for root, _, files in os.walk(EXTRACT):
        if "data.yaml" in files:
            data_dir = root
            break
else:
    data_dirs = [r for r, _, files in os.walk("/kaggle/input") if "data.yaml" in files]
    print("已解压含 data.yaml 的目录:", data_dirs)
    exam_dirs = [d for d in data_dirs if "exam" in d.lower()]
    data_dir = (exam_dirs or data_dirs)[0]
    print("考试集目录(已解压):", data_dir)

assert data_dir, "找不到考试集 data.yaml！请确认已挂载 tud-v1-1-exam200-colab-zip"

# /kaggle/input 是只读的，把考试集复制到可写的 working 目录再补空 train
import shutil
if data_dir.startswith("/kaggle/input"):
    EXTRACT = "/kaggle/working/exam_data"
    if os.path.exists(EXTRACT):
        os.system(f"rm -rf {EXTRACT}")
    shutil.copytree(data_dir, EXTRACT)
    data_dir = EXTRACT
    print("考试集工作目录:", data_dir)

# ultralytics 校验数据时会检查 train 路径存在，考试集只有 val，补一个空 train 目录
os.makedirs(os.path.join(data_dir, "images", "train"), exist_ok=True)

with open(os.path.join(data_dir, "data.yaml"), encoding="utf-8") as f:
    inner = yaml.safe_load(f)
names = inner.get("names")
assert names, "data.yaml 缺少 names"

DATA_YAML = "/kaggle/working/exam.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    f.write(f"path: {data_dir}\ntrain: images/train\nval: images/val\nnames:\n")
    if isinstance(names, dict):
        for k, v in names.items():
            f.write(f"  {k}: {v}\n")
    else:
        for i, n in enumerate(names):
            f.write(f"  {i}: {n}\n")
print("数据配置:", DATA_YAML)


## 二、逐个模型考试（挂载几个 .pt 就考几个）


In [ ]:
from ultralytics import YOLO

cands_pt = sorted(glob.glob("/kaggle/input/**/*.pt", recursive=True))
print("待考模型:")
for p in cands_pt:
    print(" ", slug_of(p), "->", p)
print()

rows = []
for p in cands_pt:
    label = slug_of(p)
    m = YOLO(p)
    r = m.val(data=DATA_YAML, split="val", verbose=False, device="0,1")
    rows.append((label, r.box.map50, r.box.map, r.box.mp, r.box.mr))
    print(f"{label:28s} mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}  P={r.box.mp:.4f}  R={r.box.mr:.4f}")

print()
print("=== 排名 (mAP50) ===")
rows.sort(key=lambda x: -x[1])
for i, (label, m50, m, p_, r_) in enumerate(rows, 1):
    print(f"第{i}名  {label:28s} mAP50={m50:.4f}")


## 三、打包考试配置（可选）


In [ ]:
import zipfile
out = "/kaggle/working/exam_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(DATA_YAML, "exam.yaml")
print("结果包:", out)
